# Week 6-7 - AlexNet From Scratch

Notebook này tiếp nối `week5_preprocess-datasets.ipynb` và tập trung vào:

- tải lại `train/val/test manifests` đã lưu ở tuần 5
- định nghĩa `PyTorch Dataset` cho Stanford Dogs
- implement AlexNet bằng `torch.nn.Module`
- train model từ random initialization
- theo dõi `loss`, `top-1`, `top-5`
- đánh giá tập test và lưu checkpoint tốt nhất


## Nguồn tham khảo kiến trúc

Tôi đã đối chiếu kiến trúc từ 2 nguồn chính:

- Paper gốc AlexNet: <https://papers.nips.cc/paper/4824-imagenet-classification-with-deep-convolutional-neural-network>
- Mã nguồn AlexNet chính thức của torchvision: <https://docs.pytorch.org/vision/2.0/_modules/torchvision/models/alexnet.html>

Ghi chú thực dụng:

- Paper gốc mô tả AlexNet với 5 convolution layers và 3 fully-connected layers ở phần classifier.
- `torchvision` cung cấp biến thể single-GPU ổn định hơn để huấn luyện trong PyTorch hiện đại.
- Notebook này **tự implement model**, nhưng follow topology thực dụng của `torchvision`:
  `64 -> 192 -> 384 -> 256 -> 256`, `AdaptiveAvgPool2d((6, 6))`, `dropout=0.5`.
- Đây là một **suy luận triển khai** dựa trên nguồn chính thức: ưu tiên tính tương thích với input `224x224` và môi trường hiện tại hơn là tái hiện nguyên xi chi tiết multi-GPU của paper 2012.


## Chuẩn bị môi trường

Notebook này giả định bạn đã chạy notebook tuần 5 để sinh ra:

- `artifacts/datasets/class_to_idx.json`
- `artifacts/datasets/train_records.json`
- `artifacts/datasets/val_records.json`
- `artifacts/datasets/test_records.json`

Nếu môi trường chưa có PyTorch:

- `uv add torch torchvision`

Notebook này hỗ trợ 2 chế độ normalization:

- `imagenet`: dùng mean/std chuẩn của ImageNet
- `dataset-specific`: tự tính mean/std từ `train_records`, rồi cache lại để tái sử dụng


In [1]:
from __future__ import annotations

import json
import random
import time
import xml.etree.ElementTree as ET
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode


In [2]:
ARTIFACTS_DIR = Path("./artifacts/datasets")
CHECKPOINT_DIR = Path("./artifacts/checkpoints")
HISTORY_DIR = Path("./artifacts/training")

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
HISTORY_DIR.mkdir(parents=True, exist_ok=True)

CLASS_TO_IDX_PATH = ARTIFACTS_DIR / "class_to_idx.json"
TRAIN_RECORDS_PATH = ARTIFACTS_DIR / "train_records.json"
VAL_RECORDS_PATH = ARTIFACTS_DIR / "val_records.json"
TEST_RECORDS_PATH = ARTIFACTS_DIR / "test_records.json"

IMAGE_SIZE = 224
RESIZE_SIZE = 256

BATCH_SIZE = 32
NUM_WORKERS = 8
NUM_EPOCHS = 20

LEARNING_RATE = 1e-2
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
DROPOUT = 0.5

USE_BBOX = False
BBOX_PADDING = 0.05

NORMALIZATION_MODE = "imagenet"  # "imagenet" | "dataset-specific"

RANDOM_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

CHECKPOINT_PATH = CHECKPOINT_DIR / "alexnet_from_scratch_best.pt"
HISTORY_PATH = HISTORY_DIR / "alexnet_from_scratch_history.json"

print(f"Device             : {DEVICE}")
print(f"Batch size         : {BATCH_SIZE}")
print(f"Num workers        : {NUM_WORKERS}")
print(f"Epochs             : {NUM_EPOCHS}")
print(f"Normalization mode : {NORMALIZATION_MODE}")
print(f"Checkpoint path    : {CHECKPOINT_PATH.resolve()}")


Device             : cuda
Batch size         : 32
Epochs             : 20
Normalization mode : imagenet
Checkpoint path    : /home/hoangdh29/fine-grained-dog-classification/artifacts/checkpoints/alexnet_from_scratch_best.pt


In [3]:
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path: Path):
    if not path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy file: {path}. Hãy chạy notebook tuần 5 trước."
        )
    return json.loads(path.read_text(encoding="utf-8"))


def parse_annotation(annotation_path: str) -> dict:
    root = ET.parse(annotation_path).getroot()
    size_node = root.find("size")
    object_node = root.find("object")
    bbox_node = object_node.find("bndbox")

    return {
        "width": int(size_node.findtext("width")),
        "height": int(size_node.findtext("height")),
        "bbox": {
            "xmin": int(bbox_node.findtext("xmin")),
            "ymin": int(bbox_node.findtext("ymin")),
            "xmax": int(bbox_node.findtext("xmax")),
            "ymax": int(bbox_node.findtext("ymax")),
        },
    }


def crop_with_bbox(
    image: Image.Image,
    bbox: dict,
    padding_ratio: float = 0.0,
) -> Image.Image:
    xmin, ymin, xmax, ymax = bbox["xmin"], bbox["ymin"], bbox["xmax"], bbox["ymax"]
    box_width = xmax - xmin
    box_height = ymax - ymin

    pad_x = int(box_width * padding_ratio)
    pad_y = int(box_height * padding_ratio)

    xmin = max(0, xmin - pad_x)
    ymin = max(0, ymin - pad_y)
    xmax = min(image.width, xmax + pad_x)
    ymax = min(image.height, ymax + pad_y)

    return image.crop((xmin, ymin, xmax, ymax))


def accuracy_at_k(logits: torch.Tensor, targets: torch.Tensor, topk=(1, 5)) -> dict[int, float]:
    max_k = min(max(topk), logits.shape[1])
    _, pred = logits.topk(max_k, dim=1, largest=True, sorted=True)
    pred = pred.t()
    correct = pred.eq(targets.view(1, -1).expand_as(pred))

    results = {}
    batch_size = targets.size(0)
    for k in topk:
        k = min(k, logits.shape[1])
        correct_k = correct[:k].reshape(-1).float().sum(0)
        results[k] = (correct_k / batch_size).item()
    return results


def count_trainable_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


seed_everything(RANDOM_SEED)


In [4]:
class_to_idx = load_json(CLASS_TO_IDX_PATH)
idx_to_class = {idx: class_name for class_name, idx in class_to_idx.items()}

train_records = load_json(TRAIN_RECORDS_PATH)
val_records = load_json(VAL_RECORDS_PATH)
test_records = load_json(TEST_RECORDS_PATH)

num_classes = len(class_to_idx)

print(f"Num classes  : {num_classes}")
print(f"Train records : {len(train_records)}")
print(f"Val records   : {len(val_records)}")
print(f"Test records  : {len(test_records)}")
print(json.dumps(train_records[0], indent=2, ensure_ascii=False))


Num classes  : 120
Train records : 14355
Val records   : 3025
Test records  : 3200
{
  "image_path": "data/stanford-dogs-dataset/2/images/Images/n02091244-Ibizan_hound/n02091244_3911.jpg",
  "annotation_path": "data/stanford-dogs-dataset/2/annotations/Annotation/n02091244-Ibizan_hound/n02091244_3911",
  "class_folder": "n02091244-Ibizan_hound",
  "breed_name": "Ibizan hound",
  "class_id": 22
}


## Dataset và transforms

Paper AlexNet dùng random crops và horizontal flips để tăng dữ liệu.
Ở đây ta dùng pipeline đơn giản, dễ tái lập:

- Train: `Resize(256) -> RandomCrop(224) -> RandomHorizontalFlip()`
- Val/Test: `Resize(256) -> CenterCrop(224)`

Ghi chú:

- paper gốc còn dùng thêm PCA color augmentation; notebook này bỏ qua để code gọn và dễ debug hơn
- `NORMALIZATION_MODE="imagenet"` sẽ dùng mean/std chuẩn của ImageNet
- `NORMALIZATION_MODE="dataset-specific"` sẽ tính mean/std từ `train_records` sau bước preprocess xác định `Resize(256) -> CenterCrop(224)` và cache vào thư mục `artifacts/datasets`


In [5]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


class StanfordDogsDataset(Dataset):
    def __init__(
        self,
        records: list[dict],
        transform=None,
        use_bbox: bool = False,
        bbox_padding: float = 0.0,
    ) -> None:
        self.records = list(records)
        self.transform = transform
        self.use_bbox = use_bbox
        self.bbox_padding = bbox_padding

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int):
        record = self.records[index]

        with Image.open(record["image_path"]) as pil_image:
            image = pil_image.convert("RGB")

        if self.use_bbox:
            annotation = parse_annotation(record["annotation_path"])
            image = crop_with_bbox(
                image,
                annotation["bbox"],
                padding_ratio=self.bbox_padding,
            )

        if self.transform is not None:
            image = self.transform(image)

        label = record["class_id"]
        return image, label


def build_stats_cache_path(
    artifacts_dir: Path,
    image_size: int,
    resize_size: int,
    use_bbox: bool,
    bbox_padding: float,
) -> Path:
    bbox_tag = "bbox" if use_bbox else "full"
    padding_tag = str(bbox_padding).replace(".", "p")
    return artifacts_dir / (
        f"train_normalization_stats_{bbox_tag}_pad-{padding_tag}_"
        f"resize-{resize_size}_crop-{image_size}.json"
    )


def compute_dataset_mean_std(
    records: list[dict],
    image_size: int,
    resize_size: int,
    use_bbox: bool = False,
    bbox_padding: float = 0.0,
) -> tuple[tuple[float, float, float], tuple[float, float, float]]:
    stats_transform = transforms.Compose(
        [
            transforms.Resize(resize_size, interpolation=InterpolationMode.BICUBIC),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
        ]
    )

    channel_sum = torch.zeros(3)
    channel_sum_sq = torch.zeros(3)
    total_pixels = 0

    for record in records:
        with Image.open(record["image_path"]) as pil_image:
            image = pil_image.convert("RGB")

        if use_bbox:
            annotation = parse_annotation(record["annotation_path"])
            image = crop_with_bbox(
                image,
                annotation["bbox"],
                padding_ratio=bbox_padding,
            )

        tensor = stats_transform(image)
        channel_sum += tensor.sum(dim=(1, 2))
        channel_sum_sq += (tensor ** 2).sum(dim=(1, 2))
        total_pixels += tensor.shape[1] * tensor.shape[2]

    mean = channel_sum / total_pixels
    std = torch.sqrt(channel_sum_sq / total_pixels - mean ** 2)

    return tuple(mean.tolist()), tuple(std.tolist())


def resolve_normalization_stats(
    mode: str,
    records: list[dict],
    artifacts_dir: Path,
    image_size: int,
    resize_size: int,
    use_bbox: bool = False,
    bbox_padding: float = 0.0,
) -> tuple[tuple[float, float, float], tuple[float, float, float]]:
    if mode == "imagenet":
        return IMAGENET_MEAN, IMAGENET_STD

    if mode != "dataset-specific":
        raise ValueError("NORMALIZATION_MODE phải là 'imagenet' hoặc 'dataset-specific'.")

    cache_path = build_stats_cache_path(
        artifacts_dir=artifacts_dir,
        image_size=image_size,
        resize_size=resize_size,
        use_bbox=use_bbox,
        bbox_padding=bbox_padding,
    )

    if cache_path.exists():
        cached = load_json(cache_path)
        mean = tuple(cached["mean"])
        std = tuple(cached["std"])
        print(f"Loaded cached dataset stats from: {cache_path}")
        return mean, std

    mean, std = compute_dataset_mean_std(
        records=records,
        image_size=image_size,
        resize_size=resize_size,
        use_bbox=use_bbox,
        bbox_padding=bbox_padding,
    )

    cache_payload = {
        "mode": mode,
        "split": "train",
        "image_size": image_size,
        "resize_size": resize_size,
        "use_bbox": use_bbox,
        "bbox_padding": bbox_padding,
        "mean": list(mean),
        "std": list(std),
    }
    cache_path.write_text(
        json.dumps(cache_payload, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    print(f"Computed and cached dataset stats to: {cache_path}")
    return mean, std


NORMALIZATION_MEAN, NORMALIZATION_STD = resolve_normalization_stats(
    mode=NORMALIZATION_MODE,
    records=train_records,
    artifacts_dir=ARTIFACTS_DIR,
    image_size=IMAGE_SIZE,
    resize_size=RESIZE_SIZE,
    use_bbox=USE_BBOX,
    bbox_padding=BBOX_PADDING,
)

print(f"Normalization mean: {NORMALIZATION_MEAN}")
print(f"Normalization std : {NORMALIZATION_STD}")

train_transform = transforms.Compose(
    [
        transforms.Resize(RESIZE_SIZE, interpolation=InterpolationMode.BICUBIC),
        transforms.RandomCrop(IMAGE_SIZE),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(mean=NORMALIZATION_MEAN, std=NORMALIZATION_STD),
    ]
)

eval_transform = transforms.Compose(
    [
        transforms.Resize(RESIZE_SIZE, interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=NORMALIZATION_MEAN, std=NORMALIZATION_STD),
    ]
)

train_dataset = StanfordDogsDataset(
    train_records,
    transform=train_transform,
    use_bbox=USE_BBOX,
    bbox_padding=BBOX_PADDING,
)

val_dataset = StanfordDogsDataset(
    val_records,
    transform=eval_transform,
    use_bbox=USE_BBOX,
    bbox_padding=BBOX_PADDING,
)

test_dataset = StanfordDogsDataset(
    test_records,
    transform=eval_transform,
    use_bbox=USE_BBOX,
    bbox_padding=BBOX_PADDING,
)

DATALOADER_KWARGS = {
    "num_workers": NUM_WORKERS,
    "pin_memory": torch.cuda.is_available(),
}
if NUM_WORKERS > 0:
    DATALOADER_KWARGS["persistent_workers"] = True
    DATALOADER_KWARGS["prefetch_factor"] = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    **DATALOADER_KWARGS,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **DATALOADER_KWARGS,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **DATALOADER_KWARGS,
)

images, labels = next(iter(train_loader))
print(f"Train batch images shape: {tuple(images.shape)}")
print(f"Train batch labels shape: {tuple(labels.shape)}")


Normalization mean: (0.485, 0.456, 0.406)
Normalization std : (0.229, 0.224, 0.225)
Train batch images shape: (32, 3, 224, 224)
Train batch labels shape: (32,)


## Implement AlexNet bằng `torch.nn.Module`

Mục tiêu ở đây là:

- **không** gọi `torchvision.models.alexnet()`
- tự viết lại toàn bộ kiến trúc
- vẫn giữ topology thực dụng giống source chính thức của `torchvision`


In [6]:
class AlexNetFromScratch(nn.Module):
    def __init__(self, num_classes: int, dropout: float = 0.5) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(64, 192, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )

        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

        self._initialize_weights()

    def _initialize_weights(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.normal_(module.weight, mean=0.0, std=0.01)
                nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.01)
                nn.init.zeros_(module.bias)

        # Theo paper gốc, một số bias được set = 1 để hỗ trợ học nhanh hơn.
        self.features[3].bias.data.fill_(1.0)   # conv2
        self.features[8].bias.data.fill_(1.0)   # conv4
        self.features[10].bias.data.fill_(1.0)  # conv5
        self.classifier[1].bias.data.fill_(1.0) # fc1
        self.classifier[4].bias.data.fill_(1.0) # fc2

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


model = AlexNetFromScratch(num_classes=num_classes, dropout=DROPOUT).to(DEVICE)
dummy_logits = model(torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE))

print(model)
print(f"Output shape       : {tuple(dummy_logits.shape)}")
print(f"Trainable params   : {count_trainable_parameters(model):,}")


AlexNetFromScratch(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, b

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    model.parameters(),
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.1,
    patience=2,
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

print(optimizer)
print(scheduler)


SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    momentum: 0.9
    nesterov: False
    weight_decay: 0.0005
)


In [8]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    scaler: torch.amp.GradScaler,
    use_amp: bool = False,
) -> dict:
    model.train()

    running_loss = 0.0
    running_top1 = 0.0
    running_top5 = 0.0
    seen_samples = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = labels.size(0)
        metrics = accuracy_at_k(logits.detach(), labels, topk=(1, 5))

        running_loss += loss.item() * batch_size
        running_top1 += metrics[1] * batch_size
        running_top5 += metrics[5] * batch_size
        seen_samples += batch_size

    return {
        "loss": running_loss / seen_samples,
        "top1": running_top1 / seen_samples,
        "top5": running_top5 / seen_samples,
    }


@torch.inference_mode()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> dict:
    model.eval()

    running_loss = 0.0
    running_top1 = 0.0
    running_top5 = 0.0
    seen_samples = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        batch_size = labels.size(0)
        metrics = accuracy_at_k(logits, labels, topk=(1, 5))

        running_loss += loss.item() * batch_size
        running_top1 += metrics[1] * batch_size
        running_top5 += metrics[5] * batch_size
        seen_samples += batch_size

    return {
        "loss": running_loss / seen_samples,
        "top1": running_top1 / seen_samples,
        "top5": running_top5 / seen_samples,
    }


## Huấn luyện

Cấu hình baseline hiện tại:

- optimizer: SGD
- loss: CrossEntropy
- scheduler: ReduceLROnPlateau theo `val_top1`
- metric chính để lưu checkpoint: `val_top1`


In [9]:
history: list[dict] = []
best_val_top1 = -float("inf")

for epoch in range(1, NUM_EPOCHS + 1):
    start_time = time.time()

    train_metrics = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=DEVICE,
        scaler=scaler,
        use_amp=USE_AMP,
    )

    val_metrics = evaluate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=DEVICE,
    )

    scheduler.step(val_metrics["top1"])
    current_lr = optimizer.param_groups[0]["lr"]
    epoch_time = time.time() - start_time

    record = {
        "epoch": epoch,
        "lr": current_lr,
        "train_loss": train_metrics["loss"],
        "train_top1": train_metrics["top1"],
        "train_top5": train_metrics["top5"],
        "val_loss": val_metrics["loss"],
        "val_top1": val_metrics["top1"],
        "val_top5": val_metrics["top5"],
        "epoch_time_sec": epoch_time,
    }
    history.append(record)

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"lr={current_lr:.5f} | "
        f"train_loss={record['train_loss']:.4f} | train_top1={record['train_top1']:.4f} | train_top5={record['train_top5']:.4f} | "
        f"val_loss={record['val_loss']:.4f} | val_top1={record['val_top1']:.4f} | val_top5={record['val_top5']:.4f} | "
        f"time={record['epoch_time_sec']:.1f}s"
    )

    if record["val_top1"] > best_val_top1:
        best_val_top1 = record["val_top1"]
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_top1": best_val_top1,
                "history": history,
                "class_to_idx": class_to_idx,
                "config": {
                    "image_size": IMAGE_SIZE,
                    "resize_size": RESIZE_SIZE,
                    "batch_size": BATCH_SIZE,
                    "num_epochs": NUM_EPOCHS,
                    "learning_rate": LEARNING_RATE,
                    "momentum": MOMENTUM,
                    "weight_decay": WEIGHT_DECAY,
                    "dropout": DROPOUT,
                    "use_bbox": USE_BBOX,
                    "bbox_padding": BBOX_PADDING,
                },
            },
            CHECKPOINT_PATH,
        )
        print(f"  Saved new best checkpoint -> {CHECKPOINT_PATH}")

    HISTORY_PATH.write_text(
        json.dumps(history, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

print(f"Best val_top1: {best_val_top1:.4f}")


Epoch 01/20 | lr=0.01000 | train_loss=4.9657 | train_top1=0.0100 | train_top5=0.0478 | val_loss=4.7834 | val_top1=0.0122 | val_top5=0.0562 | time=225.2s
  Saved new best checkpoint -> artifacts/checkpoints/alexnet_from_scratch_best.pt
Epoch 02/20 | lr=0.01000 | train_loss=4.7836 | train_top1=0.0116 | train_top5=0.0557 | val_loss=4.7808 | val_top1=0.0122 | val_top5=0.0562 | time=115.0s
Epoch 03/20 | lr=0.01000 | train_loss=4.7818 | train_top1=0.0121 | train_top5=0.0549 | val_loss=4.7796 | val_top1=0.0122 | val_top5=0.0562 | time=147.6s
Epoch 04/20 | lr=0.00100 | train_loss=4.7812 | train_top1=0.0122 | train_top5=0.0547 | val_loss=4.7790 | val_top1=0.0122 | val_top5=0.0562 | time=118.0s
Epoch 05/20 | lr=0.00100 | train_loss=4.7794 | train_top1=0.0123 | train_top5=0.0559 | val_loss=4.7790 | val_top1=0.0122 | val_top5=0.0562 | time=124.1s
Epoch 06/20 | lr=0.00100 | train_loss=4.7793 | train_top1=0.0123 | train_top5=0.0550 | val_loss=4.7789 | val_top1=0.0122 | val_top5=0.0562 | time=138.6s


In [ ]:
if not history:
    history = load_json(HISTORY_PATH)

epochs = [record["epoch"] for record in history]

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, [record["train_loss"] for record in history], label="train_loss")
plt.plot(epochs, [record["val_loss"] for record in history], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curves")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, [record["train_top1"] for record in history], label="train_top1")
plt.plot(epochs, [record["val_top1"] for record in history], label="val_top1")
plt.plot(epochs, [record["train_top5"] for record in history], label="train_top5")
plt.plot(epochs, [record["val_top5"] for record in history], label="val_top5")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Top-1 / Top-5 Accuracy Curves")
plt.legend()

plt.tight_layout()
plt.show()


## Đánh giá trên test set

Sau khi train xong, ta load lại checkpoint tốt nhất theo `val_top1` để đo trên tập test.


In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])

test_metrics = evaluate(
    model=model,
    loader=test_loader,
    criterion=criterion,
    device=DEVICE,
)

print(f"Best checkpoint epoch: {checkpoint['epoch']}")
print(f"Stored best val_top1 : {checkpoint['best_val_top1']:.4f}")
print(f"Test loss            : {test_metrics['loss']:.4f}")
print(f"Test top-1 accuracy  : {test_metrics['top1']:.4f}")
print(f"Test top-5 accuracy  : {test_metrics['top5']:.4f}")


In [ ]:
def denormalize_image(image_tensor: torch.Tensor) -> np.ndarray:
    mean = torch.tensor(NORMALIZATION_MEAN).view(3, 1, 1)
    std = torch.tensor(NORMALIZATION_STD).view(3, 1, 1)

    image = image_tensor.detach().cpu() * std + mean
    image = image.clamp(0.0, 1.0)
    return image.permute(1, 2, 0).numpy()


@torch.inference_mode()
def visualize_predictions(
    model: nn.Module,
    dataset: Dataset,
    idx_to_class: dict[int, str],
    device: torch.device,
    num_samples: int = 8,
) -> None:
    model.eval()

    chosen_indices = random.sample(range(len(dataset)), k=min(num_samples, len(dataset)))
    cols = 4
    rows = int(np.ceil(len(chosen_indices) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 4.2 * rows))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, index in zip(axes, chosen_indices):
        image_tensor, label = dataset[index]
        logits = model(image_tensor.unsqueeze(0).to(device))
        prediction = int(logits.argmax(dim=1).item())

        true_name = idx_to_class[label].split("-", 1)[1].replace("_", " ")
        pred_name = idx_to_class[prediction].split("-", 1)[1].replace("_", " ")
        correct = prediction == label

        ax.imshow(denormalize_image(image_tensor))
        ax.set_title(
            f"True: {true_name}\nPred: {pred_name}",
            color="green" if correct else "red",
            fontsize=10,
        )

    plt.tight_layout()
    plt.show()


visualize_predictions(model, test_dataset, idx_to_class, DEVICE, num_samples=8)


## Gợi ý bước tiếp theo

- Tăng số epoch lên `30-50` nếu GPU cho phép
- So sánh `USE_BBOX=False` và `USE_BBOX=True`
- So sánh AlexNet với một baseline mạnh hơn như ResNet-18 hoặc EfficientNet-B0
- Ghi thêm `classification report` hoặc `confusion matrix` cho các giống dễ nhầm
